# Measurements

Find recorded measurements, fetch their data, and manage local measurement metadata. Run **Setup**, then the section you need; no section depends on another notebook or on a previous query result. Optional cells start with `run_... = False`: configure the inputs and set the switch to `True` to run. Reset it to `False` before using Run All. Sphinx and Dash display saved outputs.

**Output examples** use shortened IDs and are not results from your target. Optional previews show the enabled operation's result; disabled cells print a skip message.

## Setup

Install `pygidata` in this kernel. Use `docs/.env.example` for `GI_BASE_URL` and credentials (`GI_TOKEN` or `GI_USER`/`GI_PASSWORD`); see Installation for opt-in loading. Set the source/measurement IDs in each example.

In [ ]:
import logging
import os
from importlib.metadata import version

import pandas as pd
from gi_data.dataclient import GIDataClient
from gi_data.mapping.models import VarSelector

base_url = os.environ["GI_BASE_URL"]
GIDataClient.set_log_level(logging.WARNING)
auth = (
    {"access_token": os.environ["GI_TOKEN"]}
    if os.getenv("GI_TOKEN")
    else {"username": os.environ["GI_USER"], "password": os.environ["GI_PASSWORD"]}
)
client = GIDataClient(base_url, **auth)
print(f"pygidata {version('pygidata')}")

## Discover history sources

Choose a source ID for the examples below. History discovery is separate from buffer discovery.

In [ ]:
pd.DataFrame([
    {"id": str(s.id), "name": s.name}
    for s in client.list_history_sources()
])

**Example output**

| id | name |
| --- | --- |
| source-1 | Test bench recordings |

Choose an `id` from your actual output for `history_source_id`.

## List a source's measurements

Set `history_source_id` below. Choose a returned ID for the read example's `measurement_id`. Bounds are Unix milliseconds.

In [ ]:
history_source_id = "your-history-source-id"
pd.DataFrame([
    {
        "id": str(m.id), "name": m.name,
        "start_ms": m.absolute_start, "end_ms": m.last_ts,
    }
    for m in client.list_history_measurements(
        history_source_id, order="DESC", limit=20,
    )
])

**Example output**

| id | name | start_ms | end_ms |
| --- | --- | ---: | ---: |
| measurement-1 | Test run | 1700000000000 | 1700000060000 |

One row per recording. Cloud may return `None` for `name`; an empty table means no measurements matched.

## Read a measurement

Set both IDs below. This resolves the selected measurement itself, then reads at most three variables from its last minute. `points` is a plotting budget, not a raw sample count.

**Cloud limitation:** `fetch_history()` uses source/time bounds, not the measurement ID as a filter. Overlapping periods are not isolated by ID. Local targets select the recording by ID.

In [ ]:
history_source_id = "your-history-source-id"
measurement_id = "your-measurement-id"
matches = client.list_history_measurements(
    history_source_id, measurements=[measurement_id], limit=1,
)
if len(matches) != 1 or str(matches[0].id) != measurement_id:
    raise ValueError("measurement_id must identify a measurement in this history source.")
measurement = matches[0]
measurement_variables = measurement.vars[:3]
if not measurement_variables:
    raise RuntimeError("The measurement has no readable variables.")
history_selectors = [
    VarSelector(SID=measurement.source_id, VID=v.id) for v in measurement_variables
]
history_end_ms = int(measurement.last_ts)
history_start_ms = max(int(measurement.absolute_start), history_end_ms - 60_000)
if history_start_ms >= history_end_ms:
    raise ValueError("The measurement has no non-empty time window.")
df_history = client.fetch_history(
    history_selectors, measurement_id=measurement.id,
    start_ms=history_start_ms, end_ms=history_end_ms, points=500,
)
if df_history.empty:
    raise RuntimeError("No samples returned for this measurement window.")
df_history = df_history.rename(columns={str(v.id): v.name for v in measurement_variables})
df_history.head()

**Example output** — cloud data, with variable names as columns:

| time | Temperature | Pressure |
| --- | ---: | ---: |
| 2023-11-14 22:13:20+00:00 | 21.5 | 1.02 |
| 2023-11-14 22:13:20.120000+00:00 | 21.7 | 1.03 |

Local HTTP uses an integer `timestamp_ns` index instead. Each column is one selected variable.

## Local-only queries

The following methods raise `NotImplementedError` on cloud. Enable these cells only on local targets; each needs Setup only.

All sources (`GET /history/structure/measurements`):

In [ ]:
run_local_list = False

if run_local_list:
    print(client.get_measurements())
else:
    print("Local query skipped. Set run_local_list = True on a local target.")

**Example output (abbreviated):**

```text
[GIHistoryMeasurement(id='measurement-1', ..., name='Test run', ...)]
```

One ID (`GET /history/structure/measurements/<id>`):

In [ ]:
run_local_get = False
measurement_id = "your-measurement-id"

if run_local_get:
    result = client.get_measurement(measurement_id)
    if result is None:
        raise ValueError("measurement_id was not found.")
    print(result)
else:
    print("Local query skipped. Set run_local_get = True on a local target.")

**Expected output:** one measurement's fields, not a list. A missing ID raises the error above.

Filter across sources (`POST /history/structure/measurements`):

In [ ]:
run_local_filter = False

if run_local_filter:
    print(client.get_measurements_advanced(
        order="DESC", limit=20,
        meas_metadata_filter=[{"Key": "TestData", "Value": "test"}],
    ))
else:
    print("Local query skipped. Set run_local_filter = True on a local target.")

**Expected output:** the same list shape as `get_measurements()`, or `[]` when nothing matches.

For a single source, pass the same filter to `list_history_measurements(source_id, ...)`. Cloud ignores metadata filters; it supports time bounds, measurement IDs, order, and limit.

## Edit metadata (local only)

**Changes target state.** Use a disposable measurement and set `measurement_id` in each cell. All management cells need Setup only and are disabled by default. When enabled, the edit/delete methods return `None`, so normally show no output; no mutation was run to produce these examples.

`POST /history/structure/measurements/<id>/metadata`:

In [ ]:
run_metadata_update = False
measurement_id = "your-measurement-id"

if run_metadata_update:
    client.add_measurement_metadata(
        measurement_id,
        meas_name="Test measurement",
        metadata=[{"Key": "TestData", "Value": "test"}],
    )
else:
    print("Metadata update skipped. Set run_metadata_update = True to change the measurement.")

## Delete metadata (local only)

**Removes metadata**, not the measurement. `DELETE /history/structure/measurements/<id>/metadata`:

In [ ]:
run_metadata_delete = False
measurement_id = "your-measurement-id"

if run_metadata_delete:
    client.delete_measurement_metadata(measurement_id)
else:
    print("Metadata deletion skipped. Set run_metadata_delete = True to remove metadata.")

## Delete a measurement (local only)

**Removes the measurement and its data.** Confirm the ID and retention requirements first. `DELETE /history/structure/measurements/<id>`:

In [ ]:
run_measurement_delete = False
measurement_id = "your-measurement-id"

if run_measurement_delete:
    client.delete_measurement(measurement_id)
else:
    print("Measurement deletion skipped. Set run_measurement_delete = True to remove it and its data.")

## Close

Run when finished, including after errors. No displayed output is expected. Re-run Setup before further API calls.

In [ ]:
client.close()